In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import syllables 

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True) 
nltk.download('stopwords', quiet=True)

print("1. Loading datasets with original index values...")
df = pd.read_csv('spotify_millsongdata.csv')

df['original_index'] = df.index 

target_artists = {
    'Pop': [
        'Rihanna',          
        'Maroon 5',         
        'Katy Perry',       
        'Justin Timberlake',
        'Britney Spears',   
        'Lady Gaga',        # 137 
        'Michael Jackson',  # 176 
        'Madonna',          # 88 
        'Bruno Mars',       # 70 
        'Kelly Clarkson',   # 157 
        'Kylie Minogue',    # 172 
        'Christina Aguilera'# 146 
    ],                      # : ~1506 

    'Rock': [
        'Queen',            # 163 
        'The Beatles',      # 178 
        'Deep Purple',      # 179 
        'Rolling Stones',   # 179 
        'Bon Jovi',         # 181 
        'Coldplay',         # 120 
        'Nirvana',          # 103 
        'Red Hot Chili Peppers', # 173 
        'Aerosmith',        # 171 
        'Scorpions'         # 167 
    ],                      # : ~1514 

    'Country': [
        'Johnny Cash',      # 183 
        'Taylor Swift',     # 81 
        'Dolly Parton',     # 180 
        'Garth Brooks',     # 85 
        'Kenny Chesney',    # 173 
        'Tim McGraw',       # 148 
        'Keith Urban',      # 110 
        'George Strait',    # 188 
        'Reba Mcentire',    # 187 
        'Vince Gill'        # 171 
    ],                      # : ~1506 

    'RnB': [
        'Diana Ross',       # 167 
        'Chris Brown',      # 145 
        'Usher',            # 117 
        'R. Kelly',         # 145 
        'Alicia Keys',      # 168  
        'Whitney Houston',  # 93 
        'Mariah Carey',     # 159 
        'Ray Charles',      # 167 
        'Stevie Wonder',    # 139 
        'Luther Vandross',  # 137 
        'Lionel Richie'     # 121 
    ],                      # : ~1458 

    'Rap': [
        'Eminem',           # 70 
        'Ice Cube',         # 80 
        'Snoop Dogg',       # 71 
        'Kanye West',       # 106 
        'Lil Wayne',        # 125 
        'LL Cool J',        # 113 
        'Ludacris',         # 106 
        'Fabolous',         # 115 
        'Insane Clown Posse',# 136 
        'Drake',            # 117 
        'Nicki Minaj',      # 88 
        'Pitbull',          # 72 
        'Outkast',          # 84 
        'Gucci Mane',       # 84 
        'Puff Daddy'        # 61 
    ]                       # : ~1508 
}

print("2. Filtering...")

filtered_dfs = []
for genre, artists in target_artists.items():
    for artist in artists:
        artist_songs = df[df['artist'] == artist].copy()
        if len(artist_songs) > 0:
            artist_songs['genre'] = genre 
            filtered_dfs.append(artist_songs)

df_subset = pd.concat(filtered_dfs, ignore_index=True)

def clean_and_tokenize(text):
    text = re.sub(r'\[.*?\]|\(.*?\)', '', str(text)) 
    text = re.sub(r'[^a-zA-Z\s]', '', text).lower()
    tokens = word_tokenize(text)
    stop_words = set(stopwords.words('english'))
    return [w for w in tokens if not w in stop_words]

def calculate_chorus_ratio(text):
    lines = [line.strip() for line in str(text).lower().split('\n') if line.strip()]
    if not lines:
        return 0.0
    line_counts = pd.Series(lines).value_counts()
    repetitive_lines = line_counts[line_counts > 1].sum()
    return round(repetitive_lines / len(lines), 4)

def calculate_syllable_density_per_line(text):
    lines = [line.strip() for line in str(text).split('\n') if line.strip()]
    if not lines:
        return 0.0

    stop_words = set(stopwords.words('english'))
    line_densities = []

    for line in lines:
        clean_line = re.sub(r'[^a-zA-Z\s]', '', line).lower()
        tokens = [w for w in word_tokenize(clean_line) if w not in stop_words]
        if not tokens:
            continue
        total_syllables = sum(syllables.estimate(word) for word in tokens)
        line_densities.append(total_syllables / len(tokens))

    if not line_densities:
        return 0.0
    return round(sum(line_densities) / len(line_densities), 4)

final_data = []
genre_counters = {g: 1 for g in target_artists.keys()}

for index, row in df_subset.iterrows():
    raw_text = row['text']
    genre = row['genre']
    artist = row['artist']
    song_name = row['song']
    orig_index = row['original_index']
    
    c_ratio = calculate_chorus_ratio(raw_text)
    
    if c_ratio > 0.0:
        s_density = calculate_syllable_density_per_line(raw_text)
        
        if s_density > 0.0:
            track_id = f"{genre}_{genre_counters[genre]:05d}"
            genre_counters[genre] += 1
            
            final_data.append({
                'original_index': orig_index,
                'track_id': track_id,
                'song_name': song_name,
                'artist': artist,
                'genre': genre,
                'lyrics': raw_text,
                'chorus_ratio': c_ratio,
                'syllable_density': s_density,
                'sentiment_score': 0.0,
                'flesch_kincaid_readability': 0.0,
                'lsa_component_1': 0.0,
                'lsa_component_2': 0.0,
                'lsa_component_3': 0.0
            })

df_final_features = pd.DataFrame(final_data)

print(f"\nRemaining number of songs after filtering: {len(df_final_features)}")

df_final_features.to_csv('billboard_lyrics_features_final.csv', index=False)
print("\nFile has been created.")

1. Loading datasets with original index values...
2. Filtering...

Remaining number of songs after filtering: 6813

File has been created.
